# A–φ・T–Ω・A–T、3次元、ゲージと境界条件

[入門](00_what_is_cln.ipynb)で見た「場から回路へ」のうち、**場をどう求めるか**を説明します。以下の記号例は3次元のポテンシャルと物理場の対応を検査するもので、3次元FEMの収束試験ではありません。

## 未知量と、比較すべき物理場

| 定式化 | 代表的な復元式 | 主に確認すること |
|---|---|---|
| A–φ | $B=\nabla\times A$, $E=-sA-\nabla\phi$ | 電流連続性、端子電圧、ゲージ |
| T–Ω | $J=\nabla\times T$, $H=H_s+T+\nabla\Omega$ | 磁束連続性、既知の励振場、循環 |
| A–T | $B=\nabla\times A$, $J=\nabla\times T$ | AmpèreとFaradayの結合残差 |

定式化の文字列が同じでも、スカラーポテンシャルの符号・係数は実装で異なります。例えば$E=-s(A+\nabla\phi_c)$なら、表の$\phi=s\phi_c$です。また、Tが定義される導体と、Ωが必要な外部領域を区別します。非単連結領域では循環自由度も指定します。

## ゲージ補正は物理場の変更ではない

A–φでは $A\mapsto A+\nabla\chi$, $\phi\mapsto\phi-s\chi$。T–Ωでは $T\mapsto T+\nabla\chi$, $\Omega\mapsto\Omega-\chi$。正しく組にして変換すれば、E・BまたはJ・Hは変わりません。

下では3次元の多項式ポテンシャルで、この不変性を記号計算します。これは任意の有限要素空間で補正が完全に表現できることを保証するものではありません。


In [1]:
import sympy as sp
x,y,z,s=sp.symbols('x y z s', real=True)
coords=(x,y,z)
def grad(f):return sp.Matrix([sp.diff(f,c) for c in coords])
def curl(v):return sp.Matrix([sp.diff(v[2],y)-sp.diff(v[1],z),sp.diff(v[0],z)-sp.diff(v[2],x),sp.diff(v[1],x)-sp.diff(v[0],y)])
A=sp.Matrix([x*y,y*z,z*x]);phi=x*y*z;chi=x**2*y+z**3
E=-s*A-grad(phi);B=curl(A)
assert sp.simplify(-s*(A+grad(chi))-grad(phi-s*chi)-E)==sp.zeros(3,1)
assert sp.simplify(curl(A+grad(chi))-B)==sp.zeros(3,1)
T=sp.Matrix([y*z,x*z,x*y**2]);Omega=x*z
assert sp.simplify(T+grad(chi)+grad(Omega-chi)-(T+grad(Omega)))==sp.zeros(3,1)
assert sp.simplify(curl(T+grad(chi))-curl(T))==sp.zeros(3,1)
print('PASS: A/phi preserves E and B; T/Omega preserves J and H')


PASS: A/phi preserves E and B; T/Omega preserves J and H


## 境界条件を比較するときは、境界項から出発する

$$\int_\Omega(\nabla\times u)\cdot v\,dV=\int_\Omega u\cdot(\nabla\times v)\,dV+\int_{\partial\Omega}(n\times u)\cdot v\,dS.$$

この境界項が、どの物理量を指定しているかを決めます。外周の開放境界、導体界面、端子励振、各段の同次補正条件、ゲージ固定は別々に記録します。Dirichlet/Neumannという名前だけを比較しません。

今回の丸線Type2では、規格化された次段の同じ解が、A法の「境界電界を維持」とT法の「境界磁界ゼロ」を同時に満たしました。**境界条件や規格化が間違っていたのではなく、未知量から場を復元する対応を追えていなかったのが説明上の問題でした。** [導出](type2_same_circuit.ipynb)

## 3次元データをどう読むか

谷本氏のA–φ・T–Ω・A–Tの研究ノートには、円柱の3次元メッシュ、HCurl/H1空間、側面・端面の拘束、勾配成分の扱いが含まれます。これを3次元再現の出発点とします。ただし元ノートの実行環境を含む公開用再現パッケージは、まだこのリリースに含めていません。

| 確認段階 | 何が分かるか |
|---|---|
| 3次元メッシュで丸線を解く | 実装された空間・境界・組立の対称ケースでの整合 |
| 軸方向に非一様な励振・形状 | φ・Ωの役割を含む非自明な場の再構成 |
| ゲージ処理・ペナルティ係数を変える | 物理場と回路定数のゲージ依存誤差 |
| メッシュ・次数を変える | 離散化誤差と定式化差の分離 |

ポテンシャル係数同士の一致ではなく、同じ座標で復元したE・B・J・H、エネルギー、端子量、残差を比較します。正の展開点の質量項が単独T空間の零空間を抑える場合でも、すべての混合空間のゲージが自動的に解決するとは限りません。

このノートで実行確認したのは上記のゲージ恒等式です。一般3次元CLNの数値検証済みという意味ではありません。


## 実際の3次元FEMを再現する

[3次元丸線の再現](06_3d_round_wire.ipynb)を追加しました。学生ノートをもとにした3定式化で、境界条件を保持し、ゲージ処理と規格化を明示して、回路定数と物理場を解析式と照合します。範囲は線形・s₀=0・導体内部・初段です。